In [ ]:
# Workbook 7/9 - Extract Final Database
#
# Purpose: This notebook produces the consolidated 'raw_db_final.csv' dataset used as the single input to
# the preprocessing and analysis notebooks.
#
# What this notebook does:
# 1) Loads the intermediate full-text database produced by earlier extraction stages.
# 2) Validates that required columns are present.
# 3) Normalizes key identifiers and fields.
# 4) Applies deterministic inclusion/exclusion rules, including:
#    - quality threshold filtering
#    - applicability filtering
#    - control match level presence (for controls)
#    - paired test/control enforcement
# 5) Writes 'raw_db_final.csv' (overwritten on each run) and prints a run summary.
#
# Inputs (Google Drive):
# - raw_fulltext_db.csv
#
# Outputs (Google Drive):
# - raw_db_final.csv
#
# Execution:
# - Run top-to-bottom.
# - Output is overwritten each run for reproducibility.
#
# Notes: This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
import os, re
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/Dissertation"
RAW_DB_CSV = os.path.join(BASE_DIR, "raw_fulltext_db.csv")
OUT_CSV    = os.path.join(BASE_DIR, "raw_db_final.csv")

# 1) Load
df0 = pd.read_csv(RAW_DB_CSV, low_memory=False)

required_cols = {
    "case_code",
    "class_label",
    "quality_percent",
    "applicability",
    "control_match_level"
}
missing = required_cols - set(df0.columns)
if missing:
    raise ValueError(f"Missing required columns in input CSV: {missing}")

# 2) Normalize
df = df0.copy()

df["case_code"] = df["case_code"].astype(str).str.strip()
df["class_label"] = df["class_label"].astype(str).str.strip().str.lower()
df["applicability"] = df["applicability"].astype(str).str.strip().str.lower()
df["quality_percent"] = pd.to_numeric(df["quality_percent"], errors="coerce")

# 3) Counters
n_origin = len(df)

# 4) Criterion 1: quality_percent > 25
mask_quality = df["quality_percent"] > 25
excluded_quality = int((~mask_quality).sum())
df = df[mask_quality]

# 4) Criterion 2: applicability == applicable
mask_applicable = df["applicability"] == "applicable"
excluded_applicability = int((~mask_applicable).sum())
df = df[mask_applicable]

# 5) Criterion 3: controls must have control_match_level
is_control = df["class_label"] == "control"
mask_control_level = ~(is_control & df["control_match_level"].isna())
excluded_control_level = int((~mask_control_level).sum())
df = df[mask_control_level]

after_basic_filters = len(df)

# 6) Pairing logic
# base_id = numeric identifier (e.g., 34500 from C34500)
df["base_id"] = df["case_code"].str.replace(r"^C(?=\d+$)", "", regex=True)

grp = df.groupby("base_id")["class_label"]
has_test = grp.transform(lambda s: (s == "test").any())
has_control = grp.transform(lambda s: (s == "control").any())

mask_paired = has_test & has_control
excluded_unpaired = int((~mask_paired).sum())

df_final = df[mask_paired].copy()

# 7) Write (overwrite every run)
os.makedirs(BASE_DIR, exist_ok=True)
df_final.to_csv(OUT_CSV, index=False)

# 8) Final counts
n_final = len(df_final)
n_test = int((df_final["class_label"] == "test").sum())
n_control = int((df_final["class_label"] == "control").sum())
n_pairs = int(df_final["base_id"].nunique())

# 9) Match-level frequency table (CONTROL cases only)
test_df = df_final[df_final["class_label"] == "control"].copy()

# Force to pandas string dtype to safely use .str
ml = test_df["control_match_level"].astype("string")

# Strip whitespace
ml = ml.str.strip()

# Mark common "missing" string tokens as <NA> without dtype downcasting
missing_tokens = {"", "nan", "NaN", "none", "None"}
ml = ml.where(~ml.isin(list(missing_tokens)), pd.NA)

# Keep only valid L-levels (L1, L1b, L2, L3...)
ml = ml.dropna()
ml = ml[ml.str.match(r"^L\d+[A-Za-z]*$", na=False)]

freq = ml.value_counts().rename_axis("match_level").reset_index(name="count")
total_ml = int(freq["count"].sum())

if total_ml > 0:
    freq["percent"] = (freq["count"] / total_ml * 100).round(2)

    # Sort nicely: L1, L1b, L2, L3...
    def _sort_key(x: str):
        m = re.match(r"^L(\d+)([A-Za-z]*)$", x)
        return (int(m.group(1)), m.group(2).lower()) if m else (10**9, x)

    freq = freq.sort_values(by="match_level", key=lambda s: s.map(_sort_key)).reset_index(drop=True)

# 10) Report
print("raw_db_final.csv written (overwritten every run)")
print("=" * 60)
print(f"Origin CSV cases:                       {n_origin:,}")
print("-" * 60)
print("Excluded by criteria:")
print(f"• quality_percent ≤ 25:                 {excluded_quality:,}")
print(f"• applicability ≠ applicable:           {excluded_applicability:,}")
print(f"• control w/ missing match level:       {excluded_control_level:,}")
print(f"• unpaired test/control cases:          {excluded_unpaired:,}")
print("-" * 60)
print(f"Cases after all filters:                {n_final:,}")
print(f"Paired base_ids:                        {n_pairs:,}")
print(f"Test cases:                             {n_test:,}")
print(f"Control cases:                          {n_control:,}")
print("=" * 60)

print("\nTEST cases — Match level frequency table (L1, L1b, L3, ...)")
print("-" * 60)
if total_ml == 0:
    print("No valid match levels found in TEST cases (after cleaning).")
else:
    print(freq.to_string(index=False))
    print("-" * 60)
    print(f"Total TEST cases with a valid match level: {total_ml:,}")